In [ ]:
# save depreciation data as individual files

import pandas as pd
import numpy as np
from statsmodels.nonparametric.smoothers_lowess import lowess
import joblib
import os

# Load data
df_2024 = pd.read_csv('../data/processed/2024_turkey_car_market_ML.csv')

# Identify all Make-Model combinations and count their occurrences
make_model_counts = df_2024.groupby(['Make', 'Model']).size().reset_index(name='counts')

# Filter out Make-Model combinations with fewer than 20 entries
make_model_filtered = make_model_counts[make_model_counts['counts'] >= 20]

# Sort the Make-Model combinations by total counts in descending order
make_model_filtered = make_model_filtered.sort_values(by='counts', ascending=False)

# Prepare a dictionary to store results
depreciation_data = {}

# Process each Make-Model combination
for _, row in make_model_filtered.iterrows():
    make, model = row['Make'], row['Model']
    print(f"Processing {make} {model}")
    
    try:
        df_make_model = df_2024[(df_2024['Make'] == make) & (df_2024['Model'] == model)]
        
        # Filter based on kilometers
        median_km = df_make_model['Kilometers (km)'].median()
        df_make_model = df_make_model[df_make_model['Kilometers (km)'] <= median_km * 1.3]
        
        # Filter out cars older than 15 years
        df_make_model = df_make_model[df_make_model['Car Age (Years)'] <= 15]

        # Ensure there are still rows left after filtering
        if df_make_model.empty:
            print(f"No data available for {make} {model} after applying filters.")
            continue

        # Group by "Car Age (Years)" and calculate the median price for each age
        median_prices = df_make_model.groupby('Car Age (Years)')['Price USD'].median().reset_index()

        # Ensure we have a complete range of years
        min_year = int(median_prices['Car Age (Years)'].min())
        max_year = int(median_prices['Car Age (Years)'].max())
        all_years = pd.DataFrame({'Car Age (Years)': range(min_year, max_year + 1)})
        median_prices = pd.merge(all_years, median_prices, on='Car Age (Years)', how='left')
        median_prices['Price USD'].interpolate(method='linear', inplace=True)

        # Add the count of cars used to calculate the median price for each year
        car_counts = df_make_model.groupby('Car Age (Years)').size().reindex(all_years['Car Age (Years)'], fill_value=0).reset_index(name='Car Count')
        median_prices = pd.merge(median_prices, car_counts, on='Car Age (Years)', how='left')

        # Filter out entries with fewer than 5 cars
        median_prices = median_prices[median_prices['Car Count'] >= 5]

        if median_prices.empty:
            print(f"No data available for {make} {model} after filtering for car count.")
            continue

        # Apply rolling average for smoothing
        median_prices['Smoothed Price'] = median_prices['Price USD'].rolling(window=3, center=True).mean()

        # Apply LOWESS for further smoothing
        lowess_smoothed = lowess(median_prices['Price USD'], median_prices['Car Age (Years)'], frac=0.2)
        median_prices['LOWESS Price'] = lowess_smoothed[:, 1]

        # Calculate the percentage loss for each year-to-year step
        median_prices['Percentage Loss'] = median_prices['LOWESS Price'].pct_change() * 100

        # Store the median prices in the results dictionary
        depreciation_data[(make, model)] = median_prices

        # Save the data
        file_path = f"../data/models/depreciation_data/{make}_{model}_depreciation.pkl"
        if not os.path.exists(file_path):
            os.makedirs(file_path)
        joblib.dump(median_prices, file_path)
        print(f"Depreciation data saved for {make} {model}")

    except Exception as e:
        print(f"Error processing {make} {model}: {e}")
        continue

# Prepare and save make-level depreciation data
make_counts = df_2024.groupby('Make').size().reset_index(name='counts')
make_filtered = make_counts[make_counts['counts'] >= 20]

for _, row in make_filtered.iterrows():
    make = row['Make']
    print(f"Processing {make}")
    
    try:
        df_make = df_2024[df_2024['Make'] == make]
        
        # Filter based on kilometers
        median_km = df_make['Kilometers (km)'].median()
        df_make = df_make[df_make['Kilometers (km)'] <= median_km * 1.3]
        
        # Filter out cars older than 15 years
        df_make = df_make[df_make['Car Age (Years)'] <= 15]

        # Ensure there are still rows left after filtering
        if df_make.empty:
            print(f"No data available for {make} after applying filters.")
            continue

        # Group by "Car Age (Years)" and calculate the median price for each age
        median_prices = df_make.groupby('Car Age (Years)')['Price USD'].median().reset_index()

        # Ensure we have a complete range of years
        min_year = int(median_prices['Car Age (Years)'].min())
        max_year = int(median_prices['Car Age (Years)'].max())
        all_years = pd.DataFrame({'Car Age (Years)': range(min_year, max_year + 1)})
        median_prices = pd.merge(all_years, median_prices, on='Car Age (Years)', how='left')
        median_prices['Price USD'].interpolate(method='linear', inplace=True)

        # Add the count of cars used to calculate the median price for each year
        car_counts = df_make.groupby('Car Age (Years)').size().reindex(all_years['Car Age (Years)'], fill_value=0).reset_index(name='Car Count')
        median_prices = pd.merge(median_prices, car_counts, on='Car Age (Years)', how='left')

        # Filter out entries with fewer than 5 cars
        median_prices = median_prices[median_prices['Car Count'] >= 5]

        if median_prices.empty:
            print(f"No data available for {make} after filtering for car count.")
            continue

        # Apply rolling average for smoothing
        median_prices['Smoothed Price'] = median_prices['Price USD'].rolling(window=3, center=True).mean()

        # Apply LOWESS for further smoothing
        lowess_smoothed = lowess(median_prices['Price USD'], median_prices['Car Age (Years)'], frac=0.2)
        median_prices['LOWESS Price'] = lowess_smoothed[:, 1]

        # Calculate the percentage loss for each year-to-year step
        median_prices['Percentage Loss'] = median_prices['LOWESS Price'].pct_change() * 100

        # Store the make-level median prices in the results dictionary
        depreciation_data[make] = median_prices

        # Save the make-level data
        file_path = f"../data/models/depreciation_data/{make}_depreciation.pkl"
        joblib.dump(median_prices, file_path)
        print(f"Depreciation data saved for {make}")

    except Exception as e:
        print(f"Error processing {make}: {e}")
        continue

# Save the full depreciation data dictionary
joblib.dump(depreciation_data, "../data/models/depreciation_data/depreciation_data_full.pkl")

In [ ]:
# Define a more refined standard depreciation rate table

'''The logic of calculating the depreciated value involves using different levels of depreciation data, starting from the most specific (exact make and model) and falling back to more general data (model-level or standard depreciation table) if necessary. Here’s how it works:

	1.	Check for Exact Make-Model Depreciation Data:
	•	First, the function checks if there is depreciation data available for the exact make and model of the car.
	•	If data is available for the specific resell age (current age of the car plus the years kept), it uses this data directly.
	•	If data for the specific resell age is not available, it estimates the depreciation rate using the average annual depreciation of the make-model.
	2.	Fallback to Model-Level Depreciation Data:
	•	If the exact make-model depreciation data is not available, the function checks if there is general depreciation data available for the model.
	•	Similar to the make-model check, if data for the specific resell age is available, it uses this data.
	•	If data for the specific resell age is not available, it estimates the depreciation rate using the average annual depreciation of the model.
	3.	Fallback to Standard Depreciation Table:
	•	If neither make-model nor model-level depreciation data is available, the function uses a predefined standard depreciation table.
	•	This table defines different depreciation rates for different ranges of car ages:
	•	10% per year for the first two years.
	•	5% per year for the next seven years.
	•	3% per year for years beyond nine.

The function then applies the determined depreciation rate to the final price of the car to calculate the depreciated value.'''

standard_depreciation_rates = {
    'first_2_years': 0.10,
    'next_7_years': 0.05,
    'after_9_years': 0.03
}

# Function to calculate depreciated value using refined standard depreciation rates
def calculate_depreciated_value(car, final_price, resell_years, depreciation_data):
    make, model = car['Make'], car['Model']
    current_age = car['Car Age (Years)']
    resell_age = current_age + resell_years

    # Check for exact make and model depreciation data
    if (make, model) in depreciation_data:
        model_data = depreciation_data[(make, model)]
        if resell_age in model_data['Car Age (Years)'].values:
            depreciation_rate = model_data.loc[model_data['Car Age (Years)'] == resell_age, 'Percentage Loss'].values[0]
            method_used = "Exact Make-Model Depreciation Data"
        else:
            # Estimate using average annual depreciation
            avg_annual_depreciation = model_data['Percentage Loss'].mean()
            depreciation_rate = avg_annual_depreciation * resell_years
            method_used = "Estimated from Make-Model Average Depreciation"
    elif model in depreciation_data:
        model_data = depreciation_data[model]
        if resell_age in model_data['Car Age (Years)'].values:
            depreciation_rate = model_data.loc[model_data['Car Age (Years)'] == resell_age, 'Percentage Loss'].values[0]
            method_used = "Make-Level Depreciation Data"
        else:
            # Estimate using average annual depreciation
            avg_annual_depreciation = model_data['Percentage Loss'].mean()
            depreciation_rate = avg_annual_depreciation * resell_years
            method_used = "Estimated from Make-Level Average Depreciation"
    else:
        # Use refined standard depreciation rates
        depreciation_rate = 0
        if resell_age <= 2:
            depreciation_rate = standard_depreciation_rates['first_2_years'] * resell_age
        elif resell_age <= 9:
            depreciation_rate = (standard_depreciation_rates['first_2_years'] * 2) + (standard_depreciation_rates['next_7_years'] * (resell_age - 2))
        else:
            depreciation_rate = (standard_depreciation_rates['first_2_years'] * 2) + (standard_depreciation_rates['next_7_years'] * 7) + (standard_depreciation_rates['after_9_years'] * (resell_age - 9))
        method_used = "Standard Depreciation Table"

    depreciated_value = final_price * (1 - depreciation_rate / 100)
    return depreciated_value, method_used